# 02 · Stint reconstruction and verification

In [ ]:
import json
import matplotlib.pyplot as plt
import pandas as pd

from boxbox.config import load_config
from boxbox.paths import artifact_paths, dataset_paths
from boxbox.preprocessing.build import read_lap_dataset

pd.set_option("display.width", 140)
pd.set_option("display.max_columns", 30)
plt.rcParams["figure.figsize"] = (9, 4)
COLOURS = {"SOFT": "#c93a55", "MEDIUM": "#b07c00", "HARD": "#3b6fb0", "UNKNOWN": "#77766f"}
subset, full = load_config("subset"), load_config("full")

Stints are rebuilt from pit stops (`stint_id` = 1 + completed stops, out-lap = stint lap 1) and checked against independent tyre changes in the tyre layer.

In [ ]:
from boxbox.reporting.eda import stint_table
data = read_lap_dataset(full)
stints = stint_table(data.laps)
stints.groupby(['stint_source', 'compound'])['length'].describe()

In [ ]:
v = data.stint_verification
print(f'Stops matched to an observed tyre change: {v.matched.mean():.1%} of {len(v)}')
v.groupby('race_id')['matched'].agg(['mean', 'size'])

Unmatched stops are typically drive-through/stop-go penalties or Ergast stops without a tyre change.

In [ ]:
v[~v.matched].merge(data.pit_stops, on=['race_id', 'driver_id', 'lap', 'stop'], how='left')

In [ ]:
fig, ax = plt.subplots()
for compound, group in stints[stints.compound.isin(['SOFT', 'MEDIUM', 'HARD'])].groupby('compound'):
    ax.hist(group['length'], bins=range(0, 50, 2), alpha=0.5, label=compound, color=COLOURS[compound])
ax.set(xlabel='stint length (laps)', ylabel='stints', title='Stint lengths by compound (2018+)'); ax.legend();

In [ ]:
sample = data.laps[(data.laps.race_id == '2024_10') & data.laps.driver_id.isin(['max_verstappen', 'norris', 'hamilton'])]
fig, ax = plt.subplots()
for (driver, stint), g in sample[sample.is_clean].groupby(['driver_id', 'stint_id']):
    ax.plot(g.stint_lap, g.lap_time_s, marker='o', ms=3, color=COLOURS[g.compound.iloc[0]], label=f'{driver} s{stint}')
ax.set(xlabel='stint lap', ylabel='lap time (s)', title='Clean lap time vs stint lap, 2024 Spanish GP'); ax.legend(fontsize=7, ncol=3);